# Papagun — загрузка Excel через pandas

Выполняйте ячейки сверху вниз в Jupyter или редакторе с поддержкой notebooks. Нужны `pandas`, `openpyxl` и Python-ядро. Для отдельного Jupyter Notebook при необходимости установите их командой `python -m pip install pandas openpyxl notebook`.

Исходные книги только читаются. Содержимое таблиц не выводится автоматически; перед сохранением ноутбука очищайте результаты ячеек, если выводили данные.

In [ ]:
from pathlib import Path

import pandas as pd

## Исходные книги

По умолчанию рабочая папка ядра — корень проекта, где лежат этот ноутбук и Excel-файлы. Если ядро запущено из другой папки, измените `DATA_DIR`.

In [ ]:
DATA_DIR = Path.cwd()
excel_files = {
    'financial': DATA_DIR / 'NL_Фин. Бюджет 2026_корректировка_15.09.2026.xlsx',
    'quantity': DATA_DIR / 'NL_Кол. Бюджет 2026 Июнь.xlsx',
}

missing_files = [key for key, path in excel_files.items() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Не найдены книги: ' + ', '.join(missing_files)
        + '. Проверьте DATA_DIR и имена файлов.'
    )

## Загрузка всех листов

`sheet_name=None` возвращает словарь листов. `header=None` сохраняет все строки, поскольку строка заголовков ещё не согласована. `dtype=object` отключает подбор типа для всей колонки; `keep_default_na=False` сохраняет текстовые маркеры вроде `NA`, а пустые ячейки обычно представлены пустой строкой. Пропуски не заменяются нулями.

Это первичное чтение значений, а не полная модель Excel: оформление и объединения не сохраняются в DataFrame, ошибки ячеек могут стать пропусками. Формулы не пересчитываются — читаются сохранённые Excel результаты, которые могут отсутствовать или быть устаревшими. Эти случаи требуют отдельной проверки перед расчётами.

In [ ]:
workbooks = {
    key: pd.read_excel(
        path,
        sheet_name=None,
        header=None,
        dtype=object,
        keep_default_na=False,
        engine='openpyxl',
    )
    for key, path in excel_files.items()
}

financial_sheets = workbooks['financial']
quantity_sheets = workbooks['quantity']

In [ ]:
sheet_summary = pd.DataFrame([
    {'book': key, 'sheet': sheet, 'rows': frame.shape[0], 'columns': frame.shape[1]}
    for key, sheets in workbooks.items()
    for sheet, frame in sheets.items()
])
sheet_summary

## Единый DataFrame филиалов

`branches_df` содержит шесть запрошенных колонок. Одна строка соответствует исходной строке статьи одного филиала за один месяц. По поручению пользователя выбран количественный бюджет (`SOURCE_BOOK = 'quantity'`). Для отдельного исследования финансового можно задать `'financial'`. Книги не смешиваются: показатели могут иметь разные единицы.

Список ниже — 20 обнаруженных листов филиалов. Сводные и вспомогательные листы, включая АУП, не входят в него. Заголовки проверяются по фактическому расположению: строка 6 — месяцы и наименование статей, строка 7 — показатели. Читается основная таблица: строки 9–193 финансовой книги или 9–141 количественной; пояснения, алгоритмы и служебный Check ниже таблиц исключены.

Итоговые строки, строки без кода и повторяющиеся названия сохраняются. Пустые значения представлены `pd.NA`, нули остаются нулями; числовые значения копируются без расчётов и округления. Ошибки Excel сохраняются текстовыми кодами (`#DIV/0!` и т. п.), поэтому колонки показателей имеют тип object. Формулы не пересчитываются: используются сохранённые результаты, которые могут быть отсутствующими или устаревшими.


In [ ]:
SOURCE_BOOK = 'quantity'  # 'quantity' — выбранный количественный бюджет
BRANCH_SHEETS = [
    'Актобе', 'Актобе АМЦ', 'Астана Биржан Сал', 'Астана Сарыарка',
    'Атырау', 'Атырау Нурсая', 'Атырау Нурсая 2', 'Караганда ЦАД',
    'Караганда Строителей', 'Кентау', 'Костанай', 'Лисаковск', 'Рудный',
    'Павлодар Дюсенова', 'Павлодар Павлова', 'Экибастуз', 'Петропавловск',
    'Усть-Каменогорск (NL UK)', 'Шымкент', 'Шымкент Абая',
]
ARTICLE_ROWS = {'financial': (9, 193), 'quantity': (9, 141)}  # номера строк Excel, включительно
MONTHS = [
    'январь', 'февраль', 'март', 'апрель', 'май', 'июнь',
    'июль', 'август', 'сентябрь', 'октябрь', 'ноябрь', 'декабрь',
]
RESULT_COLUMNS = [
    'Наименование филиала', 'Наименование статей', 'Месяц',
    'Утверждено', 'Факт по норме', 'Факт по 1С',
]


In [ ]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter


def header_text(value):
    # Нормализация применяется только к заголовкам для их проверки.
    return ' '.join(str(value).split()).casefold()


def build_branches_dataframe(source_book):
    if source_book not in ARTICLE_ROWS:
        raise ValueError("Источник должен быть 'financial' или 'quantity'.")
    sheets = workbooks[source_book]
    missing = [sheet for sheet in BRANCH_SHEETS if sheet not in sheets]
    if missing:
        raise ValueError('Не найдены листы филиалов: ' + ', '.join(missing))

    first_row, last_row = ARTICLE_ROWS[source_book]
    records, origins, issues = [], [], []
    # pandas превращает ошибки ячеек Excel в NaN. Второе чтение позволяет
    # восстановить коды ошибок, не смешивая их с пустыми значениями.
    cached_book = load_workbook(excel_files[source_book], read_only=True, data_only=True)
    try:
        for branch in BRANCH_SHEETS:
            frame = sheets[branch]
            if frame.shape[0] < last_row or frame.shape[1] < 56:
                raise ValueError(f'Недостаточный размер основной таблицы: {branch}')
            if header_text(frame.iat[5, 2]) != 'наименование статей':
                raise ValueError(f'Не найден заголовок статей в C6: {branch}')

            month_columns = {}
            for column, value in enumerate(frame.iloc[5]):
                month = header_text(value)
                if month not in MONTHS:
                    continue
                if month in month_columns:
                    raise ValueError(f'Повторяющийся заголовок месяца: {branch}, {month}')
                expected = ['утвержд.', 'факт по норме', 'факт по 1с']
                actual = [header_text(frame.iat[6, column + offset]) for offset in range(3)]
                if actual != expected:
                    raise ValueError(f'Неожиданные показатели месяца: {branch}, {month}')
                month_columns[month] = column
            if set(month_columns) != set(MONTHS):
                raise ValueError(f'Ожидаются все 12 месяцев: {branch}')

            cached_rows = cached_book[branch].iter_rows(
                min_row=first_row, max_row=last_row, max_col=frame.shape[1],
            )
            for excel_row, cells in enumerate(cached_rows, start=first_row):
                article = frame.iat[excel_row - 1, 2]
                if cells[2].data_type == 'e':
                    raise ValueError(f'Ошибка Excel вместо названия статьи: {branch}, C{excel_row}')
                if pd.isna(article) or article == '' or (isinstance(article, str) and not article.strip()):
                    continue
                if not isinstance(article, str):
                    raise ValueError(f'Название статьи не является текстом: {branch}, C{excel_row}')
                for month in MONTHS:
                    column = month_columns[month]
                    values = []
                    coordinates = []
                    for offset, metric in enumerate(RESULT_COLUMNS[3:]):
                        coordinate = f'{get_column_letter(column + offset + 1)}{excel_row}'
                        coordinates.append(coordinate)
                        cell = cells[column + offset]
                        value = frame.iat[excel_row - 1, column + offset]
                        if cell.data_type == 'e':
                            value = cell.value
                            issues.append({
                                'Строка DataFrame': len(records), 'Филиал': branch,
                                'Месяц': month, 'Показатель': metric,
                                'Ячейка': coordinate, 'Ошибка Excel': value,
                            })
                        elif pd.isna(value) or value == '':
                            value = pd.NA
                        values.append(value)
                    records.append([branch, article, month, *values])
                    origins.append([source_book, branch, excel_row, *coordinates])
    finally:
        cached_book.close()

    result = pd.DataFrame(records, columns=RESULT_COLUMNS, dtype=object)
    provenance = pd.DataFrame(origins, columns=[
        'Книга', 'Лист', 'Строка Excel',
        'Ячейка Утверждено', 'Ячейка Факт по норме', 'Ячейка Факт по 1С',
    ])
    diagnostics = pd.DataFrame(issues, columns=[
        'Строка DataFrame', 'Филиал', 'Месяц', 'Показатель', 'Ячейка', 'Ошибка Excel',
    ])
    return result, provenance, diagnostics


branches_df, branches_provenance, branches_diagnostics = build_branches_dataframe(SOURCE_BOOK)


In [ ]:
# Только структура результата: значения показателей автоматически не выводятся.
branches_summary = (
    branches_df.groupby('Наименование филиала', sort=False)
    .agg(Строк=('Месяц', 'size'), Месяцев=('Месяц', 'nunique'))
    .reset_index()
)
print(f'Филиалов: {len(branches_summary)}, строк: {len(branches_df)}, ошибок Excel: {len(branches_diagnostics)}')
branches_summary


## Доступ к результату

Единая таблица — `branches_df`. Для локального просмотра выполните `branches_df.head()` в отдельной ячейке. Например, выбор филиала: `branches_df.loc[branches_df['Наименование филиала'] == 'Актобе']`.

`branches_provenance` имеет тот же индекс, что и `branches_df`, и позволяет проверить исходные строки и ячейки. `branches_diagnostics` перечисляет ошибки Excel: их нужно разобрать до расчётов. Повторяющиеся названия и итоговые строки не суммируйте без согласованного правила. Единицы и точность финансовых расчётов остаются открытыми.

Для другой книги измените `SOURCE_BOOK` и повторно выполните ячейки формирования и сводки. При изменении структуры источников проверьте список филиалов и границы `ARTICLE_ROWS`. Перед сохранением очищайте результаты ячеек с финансовыми данными.
